# Deterministic Audit Replay

Every SCPN Phase Orchestrator simulation step is hash-chained in a JSONL audit
trail. This notebook demonstrates:

1. Running a short simulation with audit logging enabled
2. Inspecting the audit trail
3. Replaying from the trail and verifying bit-exact determinism

The replay engine recomputes each step from the logged initial conditions and
compares the resulting phase vector against the next logged entry.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

from scpn_phase_orchestrator.audit.logger import AuditLogger
from scpn_phase_orchestrator.audit.replay import ReplayEngine
from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.metrics import LayerState, UPDEState
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

## 1. Run a short simulation with audit logging

In [ ]:
N_OSC = 8
DT = 0.01
STEPS = 50

rng = np.random.default_rng(7)
omegas = rng.uniform(0.8, 1.2, N_OSC)
coupling = CouplingBuilder().build(N_OSC, base_strength=0.5, decay_alpha=0.2)
alpha = coupling.alpha.copy()
phases = rng.uniform(0, 2 * np.pi, N_OSC)

tmpdir = Path(tempfile.mkdtemp())
audit_path = tmpdir / "audit.jsonl"

logger = AuditLogger(path=audit_path)
logger.log_header(n_oscillators=N_OSC, dt=DT, method="euler", seed=7)

engine = UPDEEngine(N_OSC, DT)

for step in range(STEPS):
    phases = engine.step(phases, omegas, coupling.knm, 0.0, 0.0, alpha)
    R, psi = compute_order_parameter(phases)
    state = UPDEState(
        layers=[LayerState(R=R, psi=psi)],
        cross_layer_alignment=np.zeros((1, 1)),
        stability_proxy=R,
        regime_id="nominal",
    )
    logger.log_step(
        step,
        state,
        actions=[],
        phases=phases,
        omegas=omegas,
        knm=coupling.knm,
        alpha=alpha,
    )

logger.close()
print(f"Logged {STEPS} steps to {audit_path}")
print(f"File size: {audit_path.stat().st_size:,} bytes")
print(f"Final R: {R:.4f}")

## 2. Inspect the audit trail

Each line is a JSON object with `step`, `phases`, `R`, and a SHA-256 `_hash`
that chains to the previous entry.

In [ ]:
replay = ReplayEngine(audit_path)
entries = replay.load()

print(f"Total entries: {len(entries)}")
header = replay.load_header(entries)
print(f"Header: {json.dumps(header, indent=2)}")

step_entries = replay.step_entries(entries)
print(f"Step entries (replayable): {len(step_entries)}")
print(f"\nFirst step keys: {list(step_entries[0].keys())}")

## 3. Verify hash chain integrity

The SHA-256 chain ensures no entry was tampered with or corrupted.

In [ ]:
valid, n_verified = ReplayEngine.verify_integrity(entries)
print(f"Hash chain valid: {valid}")
print(f"Entries verified: {n_verified}")

## 4. Replay and verify determinism

The `verify_determinism_chained` method re-runs the integration from each
logged phase vector and checks that the result matches the next logged entry.

In [ ]:
replay_engine = replay.build_engine(header)
passed, n_verified = replay.verify_determinism_chained(
    replay_engine, entries, atol=1e-10
)
print(f"Determinism verified: {passed}")
print(f"Chained steps verified: {n_verified}")

## Summary

The audit trail provides:

- **Reproducibility**: any run can be replayed from its JSONL log
- **Integrity**: SHA-256 hash chain detects tampering or corruption
- **Debugging**: phase vectors at every step enable post-hoc analysis

Use `spo replay --verify <audit.jsonl>` from the CLI for the same check.